In [1]:
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns
from polars import selectors as cs
from sklearn.model_selection import train_test_split
from typing import cast

sys.path.append(r"../baseline_v2")
from add_features import add_modified_features
from load_data import load_data
from preprocess import build_preprocessor, build_target_transformer

%matplotlib inline

warnings.filterwarnings('ignore')

## Data Preprocessing

### load

In [2]:
train, _ = load_data()
train.shape

(1458, 80)

### prepare

In [3]:
X = pl.DataFrame(train)
X = add_modified_features(X)
y = X.pop('SalePrice')
tree_preprocessor, reg_preprocessor = build_preprocessor()
transformed_X = pd.DataFrame(
    data = reg_preprocessor.fit_transform(X,y),  # type:ignore
    columns= reg_preprocessor.get_feature_names_out()
)

In [12]:

from sklearn.decomposition import PCA
components = 32
pca = PCA(n_components=components)
pca.fit(transformed_X)
pd.DataFrame(data=pca.components_.T, columns=[f"PC{i}" for i in range(components)])

,PC0,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,...,PC22,PC23,PC24,PC25,PC26,PC27,PC28,PC29,PC30,PC31
0,0.004365,-0.000147,0.027132,0.010950,-0.020566,0.008641,0.005944,-0.006530,-0.008246,0.032052,...,-0.013029,0.003428,-0.014054,-0.063185,-0.015127,0.001118,-0.075393,0.031614,0.017440,0.055448
1,0.000517,0.000511,-0.000255,-0.002968,-0.001200,0.005059,0.002543,0.000675,-0.004186,0.000468,...,-0.003577,0.004251,0.000423,-0.005037,-0.000436,-0.000169,-0.001666,-0.000314,-0.009228,0.003645
2,0.005461,0.003256,0.021049,0.009723,-0.025646,0.003814,0.016973,0.011415,-0.001433,0.032492,...,-0.006268,0.011721,-0.046839,-0.093172,0.000296,0.002764,-0.067641,-0.012022,-0.007575,0.003368
3,0.001367,0.000583,0.006357,0.001140,-0.008224,0.002733,0.005536,0.004739,0.005046,0.012743,...,-0.000322,0.011424,-0.012652,-0.026473,0.002286,0.014652,-0.017897,-0.009724,-0.009116,0.013144
4,0.006228,-0.000211,0.022864,0.016280,-0.039303,-0.010540,0.027690,0.005760,0.014180,-0.000202,...,-0.013192,0.058671,-0.060723,-0.099218,0.018569,-0.018169,-0.035771,0.008714,-0.028754,0.035018
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,0.003516,-0.036421,0.163623,0.010523,0.076283,0.009343,-0.032172,-0.046908,-0.103457,-0.003467,...,0.153479,0.123323,-0.127628,0.091304,-0.102380,0.067313,-0.059833,-0.027437,0.097815,0.080212
71,-0.003382,-0.014730,-0.001700,0.001390,0.029938,-0.017188,-0.008303,0.004136,-0.024050,-0.024557,...,0.003991,0.000659,0.005255,0.033233,-0.035985,0.013301,-0.017962,-0.001874,-0.006421,0.007666
72,-0.009478,-0.007622,-0.067982,-0.043008,0.053674,0.022863,-0.120659,-0.070454,-0.060281,-0.015196,...,0.004768,-0.160076,0.128404,0.061473,-0.070669,0.036309,0.014173,-0.013467,0.039963,-0.036205
73,0.006404,-0.001472,0.047316,-0.011471,-0.018275,0.045293,0.007823,0.003636,-0.131865,-0.056378,...,0.340073,0.078586,-0.107093,0.162995,-0.202323,-0.017041,0.115701,-0.249279,-0.132271,-0.028816


In [13]:
contribution_ratios = pd.DataFrame(pca.explained_variance_ratio_)
cumulative_contribution_ratios = contribution_ratios.cumsum()
cumulative_contribution_ratios

,0
0,0.604363
1,0.868457
2,0.893741
3,0.909515
4,0.921113
5,0.930919
6,0.939470
7,0.946516
8,0.952752
9,0.958516
